In [4]:
# Логистическая регрессия (Logit vs Probit)
# Цель: Сравнить бинарную и мультиномиальную классификацию, сравнить Logit и Probit

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('kc_house_data.csv')

median_price = df['price'].median()
df['expensive'] = (df['price'] > median_price).astype(int)
print(f"Медианная цена: ${median_price:,.0f}")
print(f"Доля дорогих домов: {df['expensive'].mean()*100:.1f}%")

features = ['sqft_living', 'bedrooms', 'bathrooms', 'floors', 'yr_built']

X = df[features]
y_binary = df['expensive']    
y_multinom = df['view']  

print(f"\nПризнаки: {features}")
print(f"Уникальные значения view: {sorted(df['view'].unique())}")

Медианная цена: $450,000
Доля дорогих домов: 49.7%

Признаки: ['sqft_living', 'bedrooms', 'bathrooms', 'floors', 'yr_built']
Уникальные значения view: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [ ]:
print("БИНАРНАЯ КЛАССИФИКАЦИЯ (Logit vs Probit)")

X_train, X_test, y_train_binary, y_test_binary, y_train_multinom, y_test_multinom = train_test_split(
    X, y_binary, y_multinom, test_size=0.3, random_state=4, stratify=y_multinom
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logit_models = {}
probit_models = {}

view_levels = sorted(df['view'].unique())

for level in view_levels:
    mask_train = y_train_multinom == level
    mask_test = y_test_multinom == level
    
    n_train = mask_train.sum()
    n_test = mask_test.sum()
    
    if n_train < 30:
        print(f"\nПропускаем view={level}: мало данных (train={n_train}, test={n_test})")
        continue
    
    print(f"\n--- view={level} (train={n_train}, test={n_test}) ---")
    
    # Logit модель (sklearn)
    X_train_level = X_train_scaled[mask_train]
    y_train_level = y_train_binary[mask_train]
    
    logit_model = LogisticRegression(random_state=42, max_iter=1000)
    logit_model.fit(X_train_level, y_train_level)
    logit_models[level] = logit_model
    
    X_test_level = X_test_scaled[mask_test]
    y_test_level = y_test_binary[mask_test]
    
    train_acc_logit = accuracy_score(y_train_level, logit_model.predict(X_train_level))
    test_acc_logit = accuracy_score(y_test_level, logit_model.predict(X_test_level))
    
    # Probit модель (statsmodels)
    X_train_level_raw = X_train[mask_train]
    X_train_level_const = sm.add_constant(X_train_level_raw)
    
    probit_model = sm.GLM(y_train_level, X_train_level_const,
                          family=sm.families.Binomial(link=sm.families.links.probit())).fit()
    probit_models[level] = probit_model
    
    # Оценка на train и test
    X_test_level_raw = X_test[mask_test]
    X_test_level_const = sm.add_constant(X_test_level_raw)
    
    train_pred_probit = (probit_model.predict(X_train_level_const) > 0.5).astype(int)
    test_pred_probit = (probit_model.predict(X_test_level_const) > 0.5).astype(int)
    
    train_acc_probit = accuracy_score(y_train_level, train_pred_probit)
    test_acc_probit = accuracy_score(y_test_level, test_pred_probit)
    
    # Сравнение
    print(f"  Logit  - Train acc: {train_acc_logit:.4f}, Test acc: {test_acc_logit:.4f}")
    print(f"  Probit - Train acc: {train_acc_probit:.4f}, Test acc: {test_acc_probit:.4f}")

БИНАРНАЯ КЛАССИФИКАЦИЯ (Logit vs Probit)

--- view=0 (train=13642, test=5847) ---
  Logit  - Train acc: 0.7430, Test acc: 0.7587
  Probit - Train acc: 0.7418, Test acc: 0.7565

--- view=1 (train=233, test=99) ---
  Logit  - Train acc: 0.8283, Test acc: 0.8990
  Probit - Train acc: 0.8283, Test acc: 0.8788

--- view=2 (train=674, test=289) ---
  Logit  - Train acc: 0.8472, Test acc: 0.7716
  Probit - Train acc: 0.8472, Test acc: 0.7682

--- view=3 (train=357, test=153) ---
  Logit  - Train acc: 0.8908, Test acc: 0.8824
  Probit - Train acc: 0.8964, Test acc: 0.8758

--- view=4 (train=223, test=96) ---
  Logit  - Train acc: 0.9686, Test acc: 0.9271
  Probit - Train acc: 0.9686, Test acc: 0.9271


In [9]:
print("МУЛЬТИНОМИАЛЬНАЯ КЛАССИФИКАЦИЯ")

multinom_model = LogisticRegression(max_iter=1000, random_state=42)
multinom_model.fit(X_train_scaled, y_train_multinom)

train_pred = multinom_model.predict(X_train_scaled)
test_pred = multinom_model.predict(X_test_scaled)

train_probs = multinom_model.predict_proba(X_train_scaled)
test_probs = multinom_model.predict_proba(X_test_scaled)

# Метрики
train_accuracy = accuracy_score(y_train_multinom, train_pred)
test_accuracy = accuracy_score(y_test_multinom, test_pred)

print(f"\nТочность на обучении: {train_accuracy*100:.2f}%")
print(f"Точность на тесте:    {test_accuracy*100:.2f}%")
print(f"Разница:              {(train_accuracy - test_accuracy)*100:.2f}%")

def compute_log_likelihood(probs, y_true):
    ll = 0
    for i, true_class in enumerate(y_true):
        prob = probs[i][true_class]
        ll += np.log(max(prob, 1e-10))
    return ll

train_ll = compute_log_likelihood(train_probs, y_train_multinom)
test_ll = compute_log_likelihood(test_probs, y_test_multinom)

print(f"\nПравдоподобие на обучении: {train_ll:.2f}")
print(f"Правдоподобие на тесте:    {test_ll:.2f}")

МУЛЬТИНОМИАЛЬНАЯ КЛАССИФИКАЦИЯ

Точность на обучении: 90.10%
Точность на тесте:    90.02%
Разница:              0.08%

Правдоподобие на обучении: -5994.87
Правдоподобие на тесте:    -2605.27


In [11]:
print("ЧАСТЬ 4: ДЕТАЛЬНЫЙ АНАЛИЗ И ВЫВОДЫ")

# Матрица ошибок для лучшей модели
print("\nМатрица ошибок (тест, мультиномиальная модель):")
cm = confusion_matrix(y_test_multinom, test_pred)
print(pd.DataFrame(cm, 
                   index=[f'Истинный {i}' for i in range(5)],
                   columns=[f'Предсказанный {i}' for i in range(5)]))

# Отчёт по каждому классу
print("\nClassification Report:")
print(classification_report(y_test_multinom, test_pred, 
                            target_names=[f'view={i}' for i in range(5)]))

# Анализ ошибок
print("АНАЛИЗ ОШИБОК МУЛЬТИНОМИАЛЬНОЙ МОДЕЛИ:")

# Какие классы путаются чаще всего?
errors = (y_test_multinom != test_pred)
error_mask = errors
if error_mask.sum() > 0:
    print(f"Всего ошибок: {error_mask.sum()} из {len(y_test_multinom)} ({error_mask.mean()*100:.1f}%)")
    
    # Найдём самые частые ошибки
    error_pairs = []
    for i in range(len(y_test_multinom)):
        if errors.iloc[i]:
            true_val = y_test_multinom.iloc[i]
            pred_val = test_pred[i]
            error_pairs.append((true_val, pred_val))
    
    from collections import Counter
    common_errors = Counter(error_pairs).most_common(3)
    print("\nСамые частые ошибки:")
    for (true_val, pred_val), count in common_errors:
        print(f"  view={true_val} → предсказано view={pred_val}: {count} раз(а)")

ЧАСТЬ 4: ДЕТАЛЬНЫЙ АНАЛИЗ И ВЫВОДЫ

Матрица ошибок (тест, мультиномиальная модель):
            Предсказанный 0  Предсказанный 1  Предсказанный 2  \
Истинный 0             5830                0                6   
Истинный 1               98                0                1   
Истинный 2              286                0                1   
Истинный 3              147                0                1   
Истинный 4               89                0                1   

            Предсказанный 3  Предсказанный 4  
Истинный 0                0               11  
Истинный 1                0                0  
Истинный 2                0                2  
Истинный 3                0                5  
Истинный 4                0                6  

Classification Report:
              precision    recall  f1-score   support

      view=0       0.90      1.00      0.95      5847
      view=1       0.00      0.00      0.00        99
      view=2       0.10      0.00      0.01       289
  

In [ ]:
print("ИТОГОВЫЕ ВЫВОДЫ")

print("1. Сравнение Logit vs Probit для бинарной классификации:")
logit_better = 0
probit_better = 0
for level in logit_models.keys():
    if level in probit_models:
        mask_test = y_test_multinom == level
        if mask_test.sum() > 0:
            X_test_level = X_test_scaled[mask_test]
            y_test_level = y_test_binary[mask_test]
            
            logit_acc = accuracy_score(y_test_level, logit_models[level].predict(X_test_level))
            
            X_test_level_raw = X_test[mask_test]
            X_test_level_const = sm.add_constant(X_test_level_raw)
            probit_pred = (probit_models[level].predict(X_test_level_const) > 0.5).astype(int)
            probit_acc = accuracy_score(y_test_level, probit_pred)
            
            if logit_acc > probit_acc:
                logit_better += 1
            elif probit_acc > logit_acc:
                probit_better += 1

if logit_better > probit_better:
    print(f"   Logit показал лучшую точность в {logit_better} из {logit_better+probit_better} случаев")
elif probit_better > logit_better:
    print(f"   Probit показал лучшую точность в {probit_better} из {logit_better+probit_better} случаев")
else:
    print(f"   Logit и Probit показали одинаковую точность")

print("\n2. Мультиномиальная классификация view:")
print(f"   Точность на тесте: {test_accuracy*100:.2f}%")
print(f"   Правдоподобие: {test_ll:.2f}")

print("\n3. Сложность задачи:")
class_distribution = df['view'].value_counts().sort_index()
print(f"   Распределение классов view: {class_distribution.to_dict()}")
print("   → Классы 3 и 4 (отличный вид) встречаются редко, поэтому модель хуже их предсказывает")

print("\n4. Рекомендации:")
print("   • Для бинарной классификации Logit и Probit дают схожие результаты")
print("   • Для мультиномиальной задачи требуется больше данных для редких классов")


ИТОГОВЫЕ ВЫВОДЫ
1. Сравнение Logit vs Probit для бинарной классификации:
   Logit показал лучшую точность в 4 из 4 случаев

2. Мультиномиальная классификация view:
   Точность на тесте: 90.02%
   Правдоподобие: -2605.27

3. Сложность задачи:
   Распределение классов view: {0: 19489, 1: 332, 2: 963, 3: 510, 4: 319}
   → Классы 3 и 4 (отличный вид) встречаются редко, поэтому модель хуже их предсказывает

4. Рекомендации:
   • Для бинарной классификации Logit и Probit дают схожие результаты
   • Для мультиномиальной задачи требуется больше данных для редких классов
   • Можно попробовать oversampling для улучшения качества на view=3,4
